# الفصل 51: الذاكرة طويلة المدى للوكلاء

> نافذة سياق LLM، مهما كانت كبيرة (حتى 1M+ token في Gemini 1.5)، محدودة وتُنسى بمجرد انتهاء الجلسة.

- **الجزء:** 10 — التطبيقات والوكلاء الذكية
- **الفصل:** 51
- **المستوى:** خبير
- **زمن القراءة:** 15 دقيقة
- **المتطلبات:** الفصل 50: معماريات الوكلاء والأنظمة متعددة الوكلاء
- **الوسوم:** Memory، MemGPT، Episodic Memory، Semantic Memory

## كيف تشغّل هذا الدفتر؟

1. **Google Colab:** ارفع الملف عبر `File → Upload notebook`، ثم `Runtime → Run all`.
2. **Kaggle:** أنشئ Notebook ثم `File → Import notebook`، ثم `Run All`.
3. شغّل خلية **إعداد البيئة** أولًا؛ ستفحص الحزم وتثبّت الناقص تلقائيًا.
4. للتدريب أو النماذج الكبيرة: Colab `Runtime → Change runtime type → T4 GPU`، وKaggle `Settings → Accelerator → GPU`.


<div dir="rtl" style="margin:2px 0 18px;line-height:2.4;">
<b>افتح هذا الدفتر على منصة مجانية:</b><br>
<a href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-10-applications-agents/ch51-long-term-memory.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#F9AB00;color:#111111;">Google Colab</a> <a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-10-applications-agents%2Fch51-long-term-memory.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#20BEFF;color:#FFFFFF;">Kaggle</a> <a href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-10-applications-agents/ch51-long-term-memory.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#579ACA;color:#FFFFFF;">Binder</a> <a href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-10-applications-agents/ch51-long-term-memory.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#6C5CE7;color:#FFFFFF;">Gradient</a> <a href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-10-applications-agents/ch51-long-term-memory.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#37474F;color:#FFFFFF;">nbviewer</a> <a href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-10-applications-agents/ch51-long-term-memory.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#24292F;color:#FFFFFF;">GitHub</a>
</div>


## ملاحظات التشغيل

- **الحزم:** لا توجد مكتبات خارجية مطلوبة.
- **مقاطع توضيحية:** بعض الشيفرة تحتوي `...` أو `NotImplementedError`؛ هذا مقصود لبيان البنية وليس خطأً في الدفتر.


<style>
/* نص عربي من اليمين إلى اليسار، مع إبقاء الشيفرة من اليسار */
.rendered_html, .jp-RenderedHTMLCommon, .text_cell_render, .markdown-body {
  direction: rtl;
  text-align: right;
}
.rendered_html pre, .rendered_html code,
.jp-RenderedHTMLCommon pre, .jp-RenderedHTMLCommon code,
.text_cell_render pre, .text_cell_render code,
.markdown-body pre, .markdown-body code {
  direction: ltr;
  text-align: left;
  unicode-bidi: embed;
}
.rendered_html table, .jp-RenderedHTMLCommon table, .text_cell_render table {
  direction: rtl;
}
</style>


In [ ]:
# === إعداد البيئة ===
# يفحص هذا القسم الحزم المطلوبة ويحاول تثبيت الناقص منها تلقائيًا.
# شغّل هذه الخلية أولًا، ثم بقية خلايا الدفتر.
import importlib.util
import subprocess
import sys

print("Python:", sys.version.split()[0])

# (اسم وحدة الاستيراد، اسم حزمة pip)
REQUIRED = [

]
AUTO_INSTALL = {"PIL", "bs4", "datasets", "faiss", "fasttext", "librosa", "networkx", "opentelemetry", "peft", "prometheus_client", "pydantic", "rank_bm25", "requests", "sentence_transformers", "sentencepiece", "soundfile", "tokenizers", "trafilatura", "transformers", "wandb", "yaml"}


def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


if not REQUIRED:
    print("لا توجد مكتبات خارجية مطلوبة؛ يكفي Python القياسي.")
else:
    missing = [(module, package) for module, package in REQUIRED if not has_module(module)]
    auto_specs = []
    manual_specs = []
    for module, package in missing:
        specs = package.split()
        if module in AUTO_INSTALL:
            auto_specs.extend(specs)
        else:
            manual_specs.extend(specs)
    auto_specs = list(dict.fromkeys(auto_specs))
    manual_specs = list(dict.fromkeys(manual_specs))

    if auto_specs:
        print("⏳ تثبيت تلقائي:", " ".join(auto_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *auto_specs])
            print("✅ تم التثبيت.")
        except Exception as exc:
            print("⚠️ تعذّر التثبيت التلقائي:", exc)
            print("شغّل يدويًا: %pip install", " ".join(auto_specs))
    if manual_specs:
        print("⚠️ حزم تحتاج تثبيتًا يدويًا:", " ".join(manual_specs))
        print("شغّل: %pip install", " ".join(manual_specs))
    if not auto_specs and not manual_specs:
        print("✅ كل الحزم المطلوبة متاحة.")

    print("\nالحالة النهائية:")
    for module, package in REQUIRED:
        print(("✅" if has_module(module) else "⚠️"), package)


<div class="notebook-actions" dir="rtl"><span class="notebook-actions__label">شغّل الدفتر:</span><a class="nb-action nb-action--colab" href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-10-applications-agents/ch51-long-term-memory.ipynb" target="_blank" rel="noopener" title="افتح في Google Colab — GPU مجاني">Google Colab</a><a class="nb-action nb-action--kaggle" href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-10-applications-agents%2Fch51-long-term-memory.ipynb" target="_blank" rel="noopener" title="افتح في Kaggle — GPU/TPU مجاني">Kaggle</a><a class="nb-action nb-action--binder" href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-10-applications-agents/ch51-long-term-memory.ipynb" target="_blank" rel="noopener" title="افتح في Binder — بيئة Jupyter مجانية">Binder</a><a class="nb-action nb-action--gradient" href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-10-applications-agents/ch51-long-term-memory.ipynb" target="_blank" rel="noopener" title="افتح في Paperspace Gradient — GPU مجاني محدود">Gradient</a><a class="nb-action nb-action--nbviewer" href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-10-applications-agents/ch51-long-term-memory.ipynb" target="_blank" rel="noopener" title="اعرض الدفتر دون تنفيذ">nbviewer</a><a class="nb-action nb-action--github" href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-10-applications-agents/ch51-long-term-memory.ipynb" target="_blank" rel="noopener" title="اعرض الملف المصدر على GitHub">GitHub</a></div>

> **ملخص الفصل**
>
> نافذة سياق LLM، مهما كانت كبيرة (حتى 1M+ token في Gemini 1.5)، محدودة وتُنسى بمجرد انتهاء الجلسة.

> **بيانات الفصل**
>
> - **الجزء:** 10 — التطبيقات والوكلاء الذكية
> - **المستوى:** خبير
> - **زمن القراءة:** 15 دقيقة
> - **المتطلبات:** الفصل 50: معماريات الوكلاء والأنظمة متعددة الوكلاء


## مخرجات التعلم

بنهاية هذا الفصل ستكون قادرًا على:

- فهم الذاكرة العرضية عبر RAG على المحادثات السابقة
- فهم الذاكرة الدلالية — استخلاص الحقائق الدائمة (نهج MemGPT/Mem0)
- فهم Memory


## 1. لماذا يهمّك هذا الموضوع؟

نافذة سياق LLM، مهما كانت كبيرة (حتى 1M+ token في Gemini 1.5)، **محدودة وتُنسى** بمجرد انتهاء الجلسة. لبناء مساعد يتذكّرك عبر أشهر من التفاعل (يعرف تفضيلاتك، يتذكّر محادثات سابقة، يتعلّم من أخطائه)، تحتاج معمارية ذاكرة مستقلة عن نافذة السياق الآنية.


## 2. الخريطة الذهنية


```text
أنواع الذاكرة في أنظمة الوكلاء (مستوحاة من علم النفس المعرفي):

الذاكرة قصيرة المدى (Working Memory)
    = نافذة السياق الحالية (المحادثة الجارية فقط)

الذاكرة العرضية (Episodic Memory)
    = سجل المحادثات والأحداث السابقة المحدَّدة زمنيًا

الذاكرة الدلالية (Semantic Memory)
    = الحقائق والمعرفة العامة المُستخلَصة والمُجرَّدة من التجارب

الذاكرة الإجرائية (Procedural Memory)
    = "كيفية" أداء المهام (قد تُمثَّل كتحديثات على الـ system prompt نفسه)
```


## 3. الشرح المتدرج

### 🟢 أ) الذاكرة العرضية عبر RAG على المحادثات السابقة


In [ ]:
import datetime

class EpisodicMemory:
    """
    ذاكرة عرضية: تخزين واسترجاع محادثات/أحداث سابقة
    تُبنى فوق نفس آليات RAG من الفصل 47، لكن المصدر هو تاريخ التفاعل
    """
    def __init__(self, embedder, vector_store):
        self.embedder = embedder
        self.vector_store = vector_store   # يمكن استخدام FAISS أو قاعدة متجهات حقيقية (Pinecone/Weaviate)
        self.memory_log = []

    def store_interaction(self, user_id: str, user_message: str, assistant_response: str):
        timestamp = datetime.datetime.now()
        memory_entry = {
            "user_id": user_id,
            "content": f"المستخدم قال: {user_message}\nالمساعد ردّ: {assistant_response}",
            "timestamp": timestamp.isoformat()
        }
        embedding = self.embedder.encode([f"passage: {memory_entry['content']}"])[0]
        self.vector_store.add(embedding, metadata=memory_entry)
        self.memory_log.append(memory_entry)

    def retrieve_relevant_memories(self, user_id: str, current_query: str, top_k=5) -> list:
        query_embedding = self.embedder.encode([f"query: {current_query}"])[0]
        results = self.vector_store.search(query_embedding, top_k=top_k * 3)
        # تصفية حسب المستخدم + ترتيب حسب الصلة والحداثة معًا
        user_memories = [r for r in results if r["metadata"]["user_id"] == user_id]
        return self._rerank_by_relevance_and_recency(user_memories)[:top_k]

    def _rerank_by_relevance_and_recency(self, memories: list, recency_weight=0.3) -> list:
        """دمج التشابه الدلالي مع حداثة الذكرى — الذكريات الأحدث أكثر صلة غالبًا"""
        now = datetime.datetime.now()
        for mem in memories:
            age_days = (now - datetime.datetime.fromisoformat(mem["metadata"]["timestamp"])).days
            recency_score = 1 / (1 + age_days)   # يتناقص مع الزمن
            mem["combined_score"] = (1 - recency_weight) * mem["score"] + recency_weight * recency_score
        return sorted(memories, key=lambda x: x["combined_score"], reverse=True)


### 🟡 ب) الذاكرة الدلالية — استخلاص الحقائق الدائمة (نهج MemGPT/Mem0)

الفرق الجوهري عن الذاكرة العرضية: بدلاً من تخزين المحادثة الخام، نُلخِّص ونُستخلِص **حقائق دائمة ومُجرَّدة** عن المستخدم.


In [ ]:
class SemanticMemoryExtractor:
    """
    نهج مستوحى من MemGPT وMem0:
    بعد كل محادثة، LLM يستخرج حقائق قابلة لإعادة الاستخدام طويل المدى
    ويُحدِّث "ملف شخصية" (profile) تراكمي للمستخدم
    """
    def __init__(self, llm):
        self.llm = llm
        self.user_profiles = {}   # {user_id: {facts: [...], preferences: [...]}}

    def extract_and_update_facts(self, user_id: str, conversation: str):
        existing_facts = self.user_profiles.get(user_id, {"facts": [], "preferences": []})

        extraction_prompt = f"""الملف الحالي للمستخدم:
الحقائق المعروفة: {existing_facts['facts']}
التفضيلات المعروفة: {existing_facts['preferences']}

المحادثة الجديدة:
{conversation}

استخرج أي حقائق جديدة أو تفضيلات جديدة عن المستخدم من هذه المحادثة فقط (معلومات دائمة ومفيدة مستقبلًا،
مثل: مهنته، اهتماماته، قيود يجب تذكّرها). إذا تناقضت معلومة جديدة مع حقيقة قديمة، أشر لذلك صراحة.

أعطِ النتيجة بصيغة JSON: {{"new_facts": [...], "new_preferences": [...], "contradictions": [...]}}"""

        result = json.loads(self.llm.generate(extraction_prompt))

        existing_facts["facts"].extend(result.get("new_facts", []))
        existing_facts["preferences"].extend(result.get("new_preferences", []))

        # معالجة التناقضات: إزالة الحقائق القديمة المتناقضة
        for contradiction in result.get("contradictions", []):
            existing_facts["facts"] = [f for f in existing_facts["facts"] if f != contradiction]

        self.user_profiles[user_id] = existing_facts
        return existing_facts

    def get_memory_context_for_prompt(self, user_id: str) -> str:
        """بناء ملخص موجز للذاكرة الدلالية لإدراجه في الـ system prompt"""
        profile = self.user_profiles.get(user_id, {"facts": [], "preferences": []})
        if not profile["facts"] and not profile["preferences"]:
            return ""
        return f"""معلومات معروفة عن هذا المستخدم:
الحقائق: {'; '.join(profile['facts'])}
التفضيلات: {'; '.join(profile['preferences'])}"""


### 🔴 ج) طبقة التعمّق: إدارة الذاكرة المحدودة — نهج MemGPT (نظام تشغيل افتراضي)

**MemGPT** يُقدِّم فكرة عبقرية: معاملة نافذة سياق LLM مثل **ذاكرة RAM محدودة** في نظام تشغيل حاسوب تقليدي، مع "ذاكرة ثانوية" (قرص صلب) غير محدودة يُدار الانتقال منها وإليها تلقائيًا.


In [ ]:
class MemGPTStyleContextManager:
    """
    تطبيق مفاهيمي لفكرة MemGPT:
    - Main Context (نافذة السياق الحالية): محدودة، "سريعة"
    - External Context (ذاكرة خارجية): غير محدودة، يُسترجَع منها عند الحاجة
    - النموذج نفسه يُقرِّر متى "يُخرج" معلومات من السياق الرئيسي لحفظ مساحة،
      ومتى "يستدعي" معلومات من الذاكرة الخارجية
    """
    def __init__(self, llm, max_context_tokens=4000,
                 episodic_memory: EpisodicMemory = None,
                 semantic_memory: SemanticMemoryExtractor = None):
        self.llm = llm
        self.max_context_tokens = max_context_tokens
        self.episodic_memory = episodic_memory
        self.semantic_memory = semantic_memory
        self.current_context = []   # الرسائل الحالية في النافذة

        # أدوات خاصة لإدارة الذاكرة، متاحة للنموذج نفسه كـ function calls
        self.memory_tools = [
            Tool("أرشفة_ذاكرة", "ينقل معلومة من السياق الحالي للذاكرة طويلة المدى",
                 self._archive_to_long_term),
            Tool("استرجاع_ذاكرة", "يبحث في الذاكرة طويلة المدى عن معلومة محدّدة",
                 self._search_long_term),
        ]

    def _estimate_tokens(self, messages: list) -> int:
        return sum(len(m["content"].split()) * 1.3 for m in messages)   # تقدير تقريبي

    def _archive_to_long_term(self, content: str) -> str:
        if self.episodic_memory:
            self.episodic_memory.store_interaction("system_archive", "", content)
        return "تمت الأرشفة بنجاح"

    def _search_long_term(self, query: str) -> str:
        if self.episodic_memory:
            results = self.episodic_memory.retrieve_relevant_memories("system_archive", query)
            return "\n".join([r["metadata"]["content"] for r in results])
        return "لا توجد ذاكرة طويلة المدى متاحة"

    def add_message(self, role: str, content: str):
        self.current_context.append({"role": role, "content": content})

        # إدارة الذاكرة التلقائية: عند اقتراب الامتلاء، أرشفة أقدم الرسائل
        if self._estimate_tokens(self.current_context) > self.max_context_tokens * 0.9:
            self._compress_and_archive_oldest()

    def _compress_and_archive_oldest(self, keep_recent=5):
        """ضغط وأرشفة أقدم جزء من المحادثة قبل حذفه من النافذة النشطة"""
        to_archive = self.current_context[:-keep_recent]
        self.current_context = self.current_context[-keep_recent:]

        summary_prompt = f"لخّص المحادثة التالية بإيجاز شديد مع حفظ أي معلومة مهمة: {to_archive}"
        summary = self.llm.generate(summary_prompt)
        self._archive_to_long_term(summary)


## 4. مثال محلول خطوة بخطوة

**المحادثة الجديدة:**

> «أنا مدير منتج في شركة ناشئة، أحب الاجتماعات صباحًا، وأكره الإيميلات الطويلة. مشروعي الحالي هو إطلاق تطبيق دفع.»

**الاستخلاص المقترح:**

| النوع | المعلومة | الثقة | مدة الصلاحية |
|---|---|---:|---|
| حقيقة | المهنة: مدير منتج | 0.95 | طويلة |
| حقيقة | الشركة: ناشئة | 0.85 | متوسطة |
| تفضيل | الاجتماعات صباحًا | 0.90 | طويلة |
| تفضيل | الإيميلات القصيرة | 0.88 | طويلة |
| حقيقة | المشروع الحالي: تطبيق دفع | 0.92 | قصيرة/متوسطة |

**معالجة تناقض:** إذا كانت الذاكرة القديمة تقول «يفضّل الاجتماعات مساءً»، فلا نُضيف التفضيل الجديد بجانبه؛ نستبدله ونسجّل سبب التغيير وتاريخه. الاحتفاظ بالتناقضين معًا يجعل النموذج حائرًا.

**حساب ميزانية السياق:** إذا كان لدينا 40 حقيقة وتفضيلًا، بمتوسط 20 token لكل عنصر، فذلك 800 token. عند كل طلب نسترجع أفضل 5 فقط (نحو 100 token) بدلًا من حقن الملف كاملًا. هذا يقلّل التكلفة ويقلّل التشويش معًا.

**قاعدة الاسترجاع:** ندمج التشابه الدلالي (0.7) مع الحداثة (0.3)، ثم نُصفّي حسب المستخدم والصلاحية. الذكرى القديمة جدًا لا تختفي تلقائيًا، لكن وزنها يقل حتى تُراجَع أو تُحذف.


## 5. نصائح وحيل

> 💡 **افصل الذاكرة الخام عن المستخلصة**: خزّن المحادثة كما هي في الذاكرة العرضية، واحفظ الحقائق المُجرَّدة في الذاكرة الدلالية. الخلط بينهما يجعل الاسترجاع غامضًا.

> 💡 **أضف درجة ثقة لكل حقيقة**: لا تتعامل مع «استخرجت معلومة» كأنها يقين. الثقة تسمح بالتراجع والحذف عند ظهور دليل أقوى.

> 💡 **طبّق سياسة نسيان**: TTL للحقائق المؤقتة (المشروع الحالي)، ومراجعة دورية للحقائق القديمة، وحذف فوري بناءً على طلب المستخدم.

> 💡 **لا تحقن كل الذاكرة في الـ prompt**: استرجع فقط ما يخدم السؤال الحالي. الذاكرة الكاملة تُشتّت النموذج وتستهلك السياق.

> 💡 **قِس أثر الذاكرة**: قارن جودة الإجابة مع الذاكرة وبدونها على نفس الأسئلة. إذا لم تتحسن، فالمشكلة في الاستخلاص أو الاسترجاع، لا في كمية التخزين.


## 6. أخطاء شائعة ومصائد

> ⚠️ **تخزين كل شيء**: كل عبارة عابرة تصبح «حقيقة». النتيجة ملف ضخم متناقض. خزّن فقط ما يُفيد مستقبلًا.

> ⚠️ **تجاهل التناقضات**: معلومة جديدة لا تلغي القديمة تلقائيًا. يجب كشف التعارض وطلب تأكيد أو تطبيق سياسة «الأحدث يفوز» مع تسجيل التاريخ.

> ⚠️ **الخصوصية كفكرة لاحقة**: بيانات صحية أو مالية أو شخصية تحتاج تشفيرًا، وتحكمًا في الوصول، وسياسة حذف واضحة. لا تبنِ ذاكرة دائمة ثم تفكر في الامتثال.

> ⚠️ **الذاكرة القديمة تُفسد الإجابة**: تفضيل عمره سنة قد لا يزال يُحقن. أضف مراجعة/انتهاء صلاحية، وامنع الذكريات منخفضة الثقة من التأثير على القرارات الحساسة.

> ⚠️ **تلوث الذاكرة (Memory Poisoning)**: مستند أو رسالة تحتوي تعليمات خبيثة قد تُخزَّن كحقيقة. تحقق من المصدر، ولا تخزّن تعليمات تنفيذية من محتوى غير موثوق.


## 7. جدول مقارنة

| نوع الذاكرة | ما تُخزِّنه | متى تُستخدَم |
|---|---|---|
| **Context Window (قصيرة المدى)** | المحادثة الجارية فقط | دائمًا، تلقائيًا |
| **Episodic Memory (عرضية)** | سجل محادثات/أحداث سابقة خام أو مُلخَّصة | استرجاع "ماذا قلنا الأسبوع الماضي؟" |
| **Semantic Memory (دلالية)** | حقائق وتفضيلات مُستخلَصة ومُجرَّدة | "ما هي مهنة هذا المستخدم؟" بسرعة دون بحث |
| **Procedural Memory (إجرائية)** | تعليمات/دروس مستفادة حول كيفية أداء المهام | تحسين سلوك الوكيل تراكميًا بمرور الوقت |

> 💡 **نصيحة عملية حاسمة**: لا تُخزّن **كل** شيء في الذاكرة الدلالية دون تمييز — هذا يُنتج "ملف شخصية" ضخمًا وفوضويًا يُربك النموذج لاحقًا. طبّق سياسة انتقاء صارمة: فقط المعلومات **القابلة لإعادة الاستخدام المتكرر** (مهنة، تفضيلات ثابتة، قيود صريحة) تستحق التخزين الدائم؛ التفاصيل العابرة تبقى في الذاكرة العرضية فقط.

> ⚠️ **خطأ شائع وخطير على الخصوصية**: تخزين معلومات حساسة (صحية، مالية، شخصية جدًا) في ذاكرة طويلة المدى دون تشفير مناسب أو آلية واضحة لحذفها بناءً على طلب المستخدم (الحق في النسيان / GDPR). يجب تصميم نظام الذاكرة مع اعتبارات الخصوصية **من البداية**، وليس كإضافة لاحقة.


## 8. ملخص أفضل الممارسات

- [ ] فصل واضح بين الذاكرة العرضية (خام) والدلالية (مُستخلَصة)
- [ ] آلية صريحة لمعالجة التناقضات (معلومة جديدة تُلغي قديمة)
- [ ] حد أقصى واضح لحجم الذاكرة المُحمَّلة في أي prompt واحد
- [ ] آلية حذف/تعديل الذاكرة بناءً على طلب المستخدم (خصوصية)
- [ ] تقييم دوري: هل الذاكرة المُسترجَعة فعلًا تُحسِّن جودة الاستجابة أم تُشتِّتها؟

---


## 9. أسئلة مقابلات

**سؤال 1: ما الفرق بين الذاكرة العرضية والدلالية؟**

> **إجابة نموذجية:** العرضية تحفظ الأحداث والمحادثات كما حدثت، مرتبطة بالزمن والسياق، وتُستخدم للإجابة على «ماذا قلنا سابقًا؟». الدلالية تحفظ الحقائق والتفضيلات المُجرَّدة من الأحداث، وتُستخدم لمعرفة «من هو المستخدم وماذا يفضّل؟». الأولى خام وكثيرة التفاصيل، والثانية مُلخَّصة وقابلة لإعادة الاستخدام. نظام ناضج يحتاج الاثنين معًا.

**سؤال 2: كيف تتعامل مع معلومة جديدة تناقض قديمة؟**

> **إجابة نموذجية:** أكشف التعارض أثناء الاستخلاص، ثم أطبّق سياسة واضحة: إما «الأحدث يفوز» مع الاحتفاظ بتاريخ التغيير، أو أطلب تأكيد المستخدم إذا كانت المعلومة حساسة. لا أحتفظ بالنسختين كأنهما صحيحتان. أسجّل سبب التغيير، وأُبقي المعلومة القديمة في سجل تاريخي غير مُحقَن في الـ prompt.

**سؤال 3: كيف تحمي خصوصية المستخدم في نظام ذاكرة؟**

> **إجابة نموذجية:** مبدأ تقليل البيانات (اجمع أقل قدر لازم)، تشفير عند التخزين والنقل، عزل بيانات المستأجرين، تحكم وصول دقيق، سجل تدقيق، وسياسة حذف/تصدير واضحة. أضيف تصنيفًا للحساسية يمنع تخزين بعض الفئات أصلًا، وأُراجع الذاكرة دوريًا. الامتثال (GDPR/CCPA) يجب أن يكون في التصميم لا في طبقة لاحقة.


## 10. تمرين عملي

**الهدف:** بناء طبقة ذاكرة طويلة المدى صغيرة واختبارها عبر 10 جلسات.

**الخطوات:**
1. استخدم SQLite للذاكرة الدلالية (حقائق/تفضيلات) وFAISS أو Chroma للذاكرة العرضية.
2. اكتب دالة استخلاص تُرجع JSON: `facts`, `preferences`, `contradictions`, `confidence`.
3. نفّذ سياسة «الأحدث يفوز» مع حفظ تاريخ التغيير.
4. نفّذ استرجاعًا يدمج التشابه والحداثة، ويعيد أفضل 5 عناصر فقط.
5. أجرِ 10 جلسات محادثة متتابعة، بعضها يغيّر تفضيلًا سابقًا.
6. اختبر 10 أسئلة تعتمد على الذاكرة، و10 أسئلة لا تحتاجها.
7. احسب: دقة الاسترجاع، عدد الحقائق الخاطئة، وحجم الـ prompt لكل طلب.

**معيار النجاح:** 8/10 إجابات صحيحة، وصفر تناقضات غير معالجة، وحجم ذاكرة محقون أقل من 300 token في المتوسط.


## 11. مراجع للتعمّق

- Packer, C., et al. (2023). *MemGPT: Towards LLMs as Operating Systems*. arXiv:2310.08560.
- Chhikara, P., et al. (2025). *Mem0: Building Production-Ready AI Agents with Scalable Long-Term Memory*. arXiv:2504.19413.
- Park, J. S., et al. (2023). *Generative Agents: Interactive Simulacra of Human Behavior*. UIST.
- Sumers, T. R., et al. (2023). *Cognitive Architectures for Language Agents*. arXiv:2309.02427.
- Liu, N. F., et al. (2023). *Lost in the Middle: How Language Models Use Long Contexts*. TACL.


## 🧪 مساحة تجربة

استخدم الخلية التالية لتجربة أفكار الفصل أو حلّ التمرين. عدّل القيم وأعد تشغيل الخلية.


In [ ]:
# 🧪 مساحة تجربة
# اكتب هنا حلّ التمرين أو أي تجربة إضافية.


---
📘 **المصدر:** كتاب *إتقان المحوّلات والنماذج اللغوية الكبيرة* — التطبيقات والوكلاء الذكية / الفصل 51: الذاكرة طويلة المدى للوكلاء.

هذا الدفتر مولّد آليًا من ملف الفصل؛ لا تعدّله يدويًا، بل عدّل المصدر ثم أعد البناء.
